In [ ]:
from IPython.display import clear_output

# Content

In this notebook we will finetune a transformer network taken from huggingface, using huggingface's Trainer API

we will train facebook's mbart model to create a two way chinese to urdu language translator.

The reason for chosing mbart as the source model is because it has already seen a lot of languages's data(50 in our case) during pre-training

you can find more information about mbart on [hugging face](https://huggingface.co/facebook/mbart-large-50)

In [ ]:
# %pip install gdown
# %pip install evaluate
# %pip install pandas
# %pip install sentencepiece
# %pip install accelerator
# %pip install protobuf==3.20.3
# %pip install matplotlib
# %pip install transformers datasets

# %pip install --disable-pip-version-check \
#     torch \
#     torchdata --quiet

# %pip install tqdm

clear_output()

In [ ]:
%pip install datasets
%pip install transformers[torch]  # [torch] because Trainer API with torch needs accelerator installed. This takes care of that. Might need a restart of the session
%pip install gdown==4.5
%pip install evaluate
%pip install rouge_score

clear_output()

In [ ]:
import math

import pandas as pd
import gdown

from datasets import load_dataset, Dataset
from transformers import GenerationConfig, TrainingArguments, Trainer, TrainerCallback
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
import torch
import time
import pandas as pd
import evaluate
from tqdm import tqdm
import json

import matplotlib.pyplot as plt

import random

## Downloading the data

In [ ]:
!gdown 1ivqxeMVKDrHtjxBf3QXLAo7T9GciqNy-  # train.csv
!gdown 1Y0Ls3Rzr9MJr07fw91mFf2iSagl5CVt5  # test.csv

In [ ]:
random.seed(7)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
train_data = pd.read_csv('train.csv', usecols=['urdu', 'chinese'])
test_data = pd.read_csv('test.csv', usecols=['urdu', 'chinese'])

In [ ]:
train_data.head()

In [ ]:
train_dataset = Dataset.from_pandas(train_data, split='train')
test_dataset = Dataset.from_pandas(test_data, split='test')

## Prepare the model and tokenizers

In [ ]:
model_name='facebook/mbart-large-50'

model = MBartForConditionalGeneration.from_pretrained(model_name)
tokenizer1 = MBart50TokenizerFast.from_pretrained(model_name, src_lang="zh_CN", tgt_lang="ur_PK")
tokenizer2 = MBart50TokenizerFast.from_pretrained(model_name, src_lang="ur_PK", tgt_lang="zh_CN")

In [ ]:
def tokenize_function(rows):

    src_language = 'chinese'
    tgt_language = 'urdu'

    if random.randint(0, 1) == 1:
        src_language, tgt_language = tgt_language, src_language

    src_tokenizer, tgt_tokenizer = [get_apt_tokenizer(language) for language in (src_language, tgt_language)]

    start_prompt = f''
    end_prompt = f''
    prompt = [start_prompt + row + end_prompt for row in rows[src_language]]
    rows['src_language'] = [src_language]*len(rows[src_language])
    rows['prompt'] = prompt
    encoding = src_tokenizer(prompt, padding="max_length", truncation=True, return_tensors="pt", max_length=256)
    rows['input_ids'], rows['attention_mask'] = encoding.input_ids, encoding.attention_mask
    rows['labels'] = tgt_tokenizer(rows[tgt_language], padding="max_length", truncation=True, return_tensors="pt", max_length=256).input_ids

    return rows

def get_apt_tokenizer(src_language):
    if src_language.lower() == 'chinese':
        return tokenizer1
    elif src_language.lower() == 'urdu':
        return tokenizer2
    else:
        raise ValueError('Invalid Language')

In [ ]:
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

## Check to see tokenizer is working correctly by encoding and decoding

In [ ]:
idx = 2501
row = tokenized_test_dataset[idx]
tokenizer = get_apt_tokenizer(row['src_language'])
# print(row)
print(f'{row["src_language"]=}')
print(row['chinese'])
print(row['urdu'])
print(tokenizer.decode(row['input_ids'], skip_special_tokens=True))
print(tokenizer.decode(row['labels'], skip_special_tokens=True))

In [ ]:
tokenized_train_dataset.shape

In [ ]:
tokenized_train_dataset = tokenized_train_dataset.shuffle(seed=13)
tokenized_test_dataset = tokenized_test_dataset.shuffle(seed=13)

In [ ]:
print(tokenized_train_dataset.shape)
tokenized_train_dataset

## Pre-trained model

In [ ]:
sample_chinese = '加强企业法治文化建设,提高经营管理人员依法经营、依法管理能力。'
with torch.no_grad():
    res = model.generate(**tokenizer1(sample_chinese, return_tensors='pt'))
tokenizer1.batch_decode(res)[0]

## Training the model

In [ ]:
# Create a list to store losses
train_losses = []
val_losses = []

# Custom callback to store losses every 500 steps
class CustomCallback(TrainerCallback):

    def on_log(self, args, state, control, **kwargs):
        # Check if training loss is available in the logs
        if "train_loss" in state.log_history:
            train_loss = state.log_history["train_loss"]
            train_losses.append(train_loss)

        # Check if validation loss is available in the logs
        if "eval_loss" in state.log_history:
            val_loss = state.log_history["eval_loss"]
            val_losses.append(val_loss)

output_dir = f'./checkpoints/mbart-large-translator-full-run'  # full for full fine tune

training_args = TrainingArguments(
    output_dir=output_dir,
    # per_device_train_batch_size=4,
    auto_find_batch_size=True,
    learning_rate=5e-5,
    num_train_epochs=2,
    eval_steps=500,
    save_steps=5000,
    logging_steps=500,
    evaluation_strategy="steps",
    save_strategy="steps",
    # load_best_model_at_end=True,
    warmup_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    # compute_metrics=lambda x: 1
    callbacks=[CustomCallback()]
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained('mbart_2ep_full_finetuned_bidir')

In [ ]:
sample_chinese = '加强企业'
with torch.no_grad():
    encoding = tokenizer1(sample_chinese, return_tensors='pt')
    input_ids, attn_mask = encoding.input_ids.to(model.device), encoding.attention_mask.to(model.device)
    res = model.generate(input_ids=input_ids, attention_mask=attn_mask)
print(tokenizer1.batch_decode(res)[0])

In [ ]:
sample_urdu = 'اہمیں ابتدائی کارکردگی پر توجہ دینی چاہیے اور ان احتیاطی تدابیر پر عمل کرنا چاہیے تاکہ مسائل سے بچا جا سکے۔'
with torch.no_grad():
    encoding = tokenizer2(sample_urdu, return_tensors='pt')
    input_ids, attn_mask = encoding.input_ids.to(model.device), encoding.attention_mask.to(model.device)
    res = model.generate(input_ids=input_ids, attention_mask=attn_mask)
print(tokenizer2.batch_decode(res)[0])

In [ ]:
plt.plot(train_losses, label='Train Losses')
plt.plot(val_losses, label='Val Losses')

plt.legend()
plt.show()

## Evaluating the results

In [ ]:
untrained_model = MBartForConditionalGeneration.from_pretrained(model_name)

In [ ]:
bleu = evaluate.load('bleu')

In [ ]:
cn_to_ur_data = test_data.sample(frac=0.5, random_state=123)
ur_to_cn_data = test_data.loc[test_data.index.difference(cn_to_ur_data.index)]

In [ ]:
batch_size = 16

num_batches = int(math.ceil(len(cn_to_ur_data)/batch_size))

all_untrained_model_outputs = []
all_trained_model_outputs = []

for i in tqdm(range(num_batches)):

    batch_start = i*batch_size
    batch_end = batch_start + batch_size

    batch_df = cn_to_ur_data.iloc[batch_start:] if i == num_batches - 1 else cn_to_ur_data.iloc[batch_start: batch_end]

    prompts = [f'{chinese}' for chinese in batch_df['chinese']]

    with torch.no_grad():

        encodings = tokenizer1(prompts, return_tensors="pt", padding="max_length", truncation=True, max_length=256)

        input_ids = encodings.input_ids.to(device)
        attention_mask = encodings.attention_mask.to(device)

        model_outputs = model.generate(input_ids=input_ids,
                                                         attention_mask=attention_mask,
                                                         generation_config=GenerationConfig(max_new_tokens=256))
        model_text_output = tokenizer1.batch_decode(model_outputs, skip_special_tokens=True)
        all_untrained_model_outputs.extend(model_text_output)

        trained_model_outputs = model.generate(input_ids=input_ids,
                                                 attention_mask=attention_mask,
                                                 generation_config=GenerationConfig(max_new_tokens=256))
        trained_model_text_output = tokenizer1.batch_decode(trained_model_outputs, skip_special_tokens=True)
        all_trained_model_outputs.extend(trained_model_text_output)

In [ ]:
cn_to_ur_data['trained_model_urdu'] = all_trained_model_outputs
cn_to_ur_data['untrained_model_urdu'] = all_untrained_model_outputs

In [ ]:
original_bleu = bleu.compute(
    predictions=cn_to_ur_data['untrained_model_urdu'],
    references=cn_to_ur_data['urdu'],
)

trained_bleu = bleu.compute(
    predictions=cn_to_ur_data['trained_model_urdu'],
    references=cn_to_ur_data['urdu'],
)

In [ ]:
print('Chinese to Urdu')
print('Original model:')
print(json.dumps(original_bleu, indent=2))
print('Trained model:')
print(json.dumps(trained_bleu, indent=2))